# Used-car data preprocessing

This notebook prepares the cleaned Hatla2ee listings for price modelling. It applies the EDA data-quality decisions, derives numeric mileage, and groups listing locations by governorate. Categorical encoding is deliberately deferred until after a train/test split to prevent leakage.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'Data' / 'clean').is_dir():
            return candidate
    raise FileNotFoundError('Could not find Data/clean/.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SNAPSHOT_DATE = '2026-10-13'  # YYYY-MM-DD; use one cleaned snapshot end-to-end
INPUT_PATH = PROJECT_ROOT / 'Data' / 'clean' / f'{SNAPSHOT_DATE}_used_cars_cleaned.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Data' / 'preprocessed'
OUTPUT_PATH = OUTPUT_DIR / f'{SNAPSHOT_DATE}_used_cars_preprocessed.csv'

raw = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(raw):,} rows and {raw.shape[1]} columns from {INPUT_PATH.name}.')
display(raw.head())


Loaded 10,819 rows and 14 columns from 2026-05-25_used_cars_cleaned.csv.


,id,title,year,ad_date,transmission,price,fingerprint,fuel,brand,model,color,class,km,city
0,7222579,Cupra Terramar 2026,2026,NaN,1,1200000,7222579-1200000,Natural Gas,Cupra,Terramar,NaN,NaN,"2,000 KM","Nasr city, Cairo"
1,7204848,Seat Tarraco 2021,2021,NaN,1,1580000,7204848-1580000,Gas,Seat,Tarraco,NaN,NaN,"59,000 KM","Nasr city, Cairo"
2,7209848,Volkswagen Passat 2022,2022,NaN,1,1830000,7209848-1830000,Gas,Volkswagen,Passat,NaN,NaN,"60,000 KM","Nasr city, Cairo"
3,7223028,Audi Q3 2024,2024,NaN,1,2590000,7223028-2590000,Gas,Audi,Q3,NaN,NaN,"14,000 KM","Smouha, Alexandria"
4,7189822,Mercedes CLS Class 2019,2019,NaN,1,3650000,7189822-3650000,Gas,Mercedes,CLS Class,NaN,NaN,"63,000 KM","Nozha, Alexandria"


## Personalised dataset

The subsample is a reproducible simple-random 80% sample drawn without replacement from the selected `YYYY-MM-DD_used_cars_cleaned.csv` snapshot, using TKH student ID **2300632** as the fixed random seed. All cleaning summaries, exported data, and later reported modelling metrics must use this personalised subsample.



In [2]:
STUDENT_ID = 2_300_632
PERSONALISED_SAMPLE_FRACTION = 0.80

personalised = (
    raw.sample(frac=PERSONALISED_SAMPLE_FRACTION, random_state=STUDENT_ID, replace=False)
    .sort_index()
    .copy()
)

assert personalised.index.is_unique
print(
    f'Personalised subsample: {len(personalised):,} of {len(raw):,} rows '
    f'({len(personalised) / len(raw):.1%}); seed/student ID = {STUDENT_ID}.'
)
display(personalised.head())


Personalised subsample: 8,655 of 10,819 rows (80.0%); seed/student ID = 2300632.


,id,title,year,ad_date,transmission,price,fingerprint,fuel,brand,model,color,class,km,city
1,7204848,Seat Tarraco 2021,2021,NaN,1,1580000,7204848-1580000,Gas,Seat,Tarraco,NaN,NaN,"59,000 KM","Nasr city, Cairo"
2,7209848,Volkswagen Passat 2022,2022,NaN,1,1830000,7209848-1830000,Gas,Volkswagen,Passat,NaN,NaN,"60,000 KM","Nasr city, Cairo"
3,7223028,Audi Q3 2024,2024,NaN,1,2590000,7223028-2590000,Gas,Audi,Q3,NaN,NaN,"14,000 KM","Smouha, Alexandria"
4,7189822,Mercedes CLS Class 2019,2019,NaN,1,3650000,7189822-3650000,Gas,Mercedes,CLS Class,NaN,NaN,"63,000 KM","Nozha, Alexandria"
5,7217240,Seat Ibiza 2025,2025,NaN,1,1270000,7217240-1270000,Gas,Seat,Ibiza,NaN,NaN,"18,000 KM",Damietta


## Cleaning decisions from EDA

The input is already deduplicated. The EDA identified zero prices, prices of at least 100 million EGP, mileage of at least one million KM, and years beyond 2026 as invalid. This notebook removes those records and rows with a missing mileage. It keeps zero-mileage records, because they may be genuine new-car listings.

The all-missing `ad_date`, `color`, and `class` columns are excluded. `fingerprint` is an identifier and `title` overlaps with brand, model, and year, so neither is included in the model-ready output. `listing_id` is retained only for traceability and must not be used as a model feature.


In [3]:
MAX_PRICE_EGP = 100_000_000
MAX_MILEAGE_KM = 1_000_000
MIN_YEAR = 1950
MAX_YEAR = 2026

working = personalised.copy()
working['mileage_km'] = pd.to_numeric(
    working['km'].astype('string').str.replace(r'[^0-9]', '', regex=True),
    errors='coerce',
)

invalid_price = working['price'].isna() | working['price'].le(0) | working['price'].ge(MAX_PRICE_EGP)
invalid_mileage = working['mileage_km'].isna() | working['mileage_km'].ge(MAX_MILEAGE_KM)
invalid_year = working['year'].isna() | ~working['year'].between(MIN_YEAR, MAX_YEAR)
valid_rows = ~(invalid_price | invalid_mileage | invalid_year)

cleaning_audit = pd.DataFrame({
    'rule': [
        'Invalid price (missing, zero, or at least 100 million EGP)',
        'Invalid mileage (missing or at least 1 million KM)',
        'Invalid model year (outside 1950–2026)',
        'Removed by one or more rules',
    ],
    'rows': [invalid_price.sum(), invalid_mileage.sum(), invalid_year.sum(), (~valid_rows).sum()],
})
cleaning_audit['share_of_input_%'] = (100 * cleaning_audit['rows'] / len(working)).round(2)
display(cleaning_audit)

cleaned = working.loc[valid_rows].copy()
print(f'Rows retained after numeric validation: {len(cleaned):,} ({len(cleaned) / len(working):.1%} of input).')


,rule,rows,share_of_input_%
0,"Invalid price (missing, zero, or at least 100 ...",10,0.12
1,Invalid mileage (missing or at least 1 million...,21,0.24
2,Invalid model year (outside 1950–2026),3,0.03
3,Removed by one or more rules,34,0.39


Rows retained after numeric validation: 8,621 (99.6% of input).


## Standardize features and derive governorate

The supplied city-to-governorate grouping is used below. Source locations are commonly stored as `district, governorate`; therefore `find_gover` checks every comma-separated component case-insensitively. Locations without a supplied mapping are preserved as `Other / Unmapped` for review rather than discarded.


In [4]:
GOVERNORATE_CITIES = {
    'Cairo': ['Cairo', 'Mohandessin', 'Nasr city', 'El Marg', 'Sheraton', 'Heliopolis', 'El Manial', 'Mokattam', 'El Gamaleya', 'Al Shorouk', 'Shobra', 'Tagamo3 - New Cairo', 'New Administrative Capital', 'Badr City', 'Zamalek', 'Ain Shams', 'Madinaty', 'Maadi', 'Warraq', 'El Katameya', 'El Salam City'],
    'Giza': ['Imbaba', 'Kerdasa', 'Dokki', 'Saft El Laban', 'El Haram', 'Sheikh Zayed City', '6 October', 'Giza', 'Pyramids Gardens'],
    'Dakahlia': ['Mit Ghamr', 'El Senbellawein', 'Dikirnis', 'Mansoura', 'Dakahlia'],
    'Red Sea': ['Ras Gharib', 'Safaga', 'Hurghada', 'El Gouna', 'Red Sea'],
    'Luxor': ['Luxor'], 'Sharkia': ['Dyarb Negm', 'Bilbeis'], 'Damietta': ['Damietta'],
    'Monufia': ['Sadat City', 'Shibin el Kom', 'El Bagour', 'Ashmoun', 'Menouf', 'Quesna', 'Monufia', 'Tala'],
    'Alexandria': ['Alexandria', 'Borg el arab', 'Amreya', 'Abu Qir', 'Agamy'],
    'Qalubia': ['Kafr Shukr', 'Banha', 'Khanka', 'Touhk', 'El Qanater El Khayreya', 'Qalyub', 'Shibin El Qanater', 'Obour City', 'Qalyubia'],
    'Kafr el-Sheikh': ['Kafr el-Sheikh'], 'El Minya': ['El Minya'],
    'ALGharbia': ['El Mahalla', 'Kafr El Zayat', 'Zefta', 'Tanta', 'Gharbia'],
    'ALSharqia': ['Faqous', 'Minya Al Qamh', 'Zagazig', 'Sharqia', 'Abu Kabir', '10th of Ramadan'],
    'Helwan': ['Helwan'], 'Aswan': ['Kom Ombo', 'Aswan'], 'Matrouh': ['Marsa Matrouh', 'El-Alamein'],
    'South Sinai': ['Sharm el-Sheikh'], 'North Sinai': ['El-Arish'], 'Sohag': ['Sohag'],
    'Ismailia': ['Ismailia'], 'Suez': ['Suez'], 'Qena': ['Qena'],
    'ElBeheira': ['Damanhur', 'Kom Hamada', 'Edku', 'Kafr el-Dawwar', 'Abu Hummus', 'Rosetta', 'Beheira'],
    'Beni Suef': ['Beni Suef'], 'El Wadi El Gedid': ['El Wadi El Gedid'],
    'Faiyum': ['Faiyum'], 'Asyut': ['Asyut'], 'Port Said': ['Port Said'],
}

CITY_TO_GOVERNORATE = {
    city.casefold(): governorate
    for governorate, cities in GOVERNORATE_CITIES.items()
    for city in cities
}

def find_gover(city: object) -> str:
    """Return a governorate for a city/district, or an empty string if unmapped."""
    if pd.isna(city):
        return ''
    for city_part in str(city).split(','):
        governorate = CITY_TO_GOVERNORATE.get(city_part.strip().casefold())
        if governorate:
            return governorate
    return ''

def normalize_text(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().replace('', pd.NA)

fuel_map = {
    'gas': 'Gas', 'diesel': 'Diesel', 'natural gas': 'Natural Gas',
    'electric': 'Electric', 'hybrid': 'Hybrid',
}

cleaned['brand'] = normalize_text(cleaned['brand'])
cleaned['model'] = normalize_text(cleaned['model'])
location = normalize_text(cleaned['city'])
cleaned['governorate'] = location.map(find_gover).replace('', 'Other / Unmapped')
cleaned['fuel'] = normalize_text(cleaned['fuel']).str.lower().map(fuel_map).fillna('Unknown')
cleaned['transmission'] = cleaned['transmission'].map({1: 'Automatic', 0: 'Manual'}).fillna('Unknown')

governorate_distribution = (
    cleaned.groupby('governorate', dropna=False).size().rename('listing_count')
    .sort_values(ascending=False).to_frame()
)
display(governorate_distribution)

unmapped_locations = (
    location.loc[cleaned['governorate'].eq('Other / Unmapped')]
    .value_counts().rename('listing_count').to_frame()
)
print(f'Unmapped listings: {unmapped_locations["listing_count"].sum() if not unmapped_locations.empty else 0:,}')
display(unmapped_locations.head(20))


,listing_count
governorate,
Cairo,4212
Giza,1670
Alexandria,640
Qalubia,285
ALSharqia,249
ALGharbia,206
Dakahlia,188
Other / Unmapped,137
Monufia,136


Unmapped listings: 137


,listing_count
city,
El Faiyum,69
Ben Suef,14
"Al Abbassia, Greater Cairo ( Cairo + Giza)",9
Honda City 2016,4
Honda City 2008,4
Honda City 2012,4
South Sinai,4
"Ashmun, Menofia",3
"Tor Sinai, South Sinai",3


## Select model-ready columns and export

The export includes the target (`price`), numerical vehicle information, standardized categorical fields, and governorate as the only location identifier. Scaling and category encoding are left to the post-split training pipeline.


In [5]:
MODEL_READY_COLUMNS = [
    'id', 'price', 'year', 'mileage_km', 'transmission', 'fuel',
    'brand', 'model', 'governorate',
]

preprocessed = (
    cleaned.loc[:, MODEL_READY_COLUMNS]
    .rename(columns={'id': 'listing_id'})
    .sort_values('listing_id')
    .reset_index(drop=True)
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
preprocessed.to_csv(OUTPUT_PATH, index=False)

print(f'Saved {len(preprocessed):,} rows × {preprocessed.shape[1]} columns to:\n{OUTPUT_PATH}')
display(preprocessed.head())
display(preprocessed.isna().sum().rename('missing_values').to_frame())


Saved 8,621 rows × 9 columns to:
/Users/yassien/Documents/Uni/MachineLearning/Project/Data/preprocessed/2026-05-25_used_cars_preprocessed.csv


,listing_id,price,year,mileage_km,transmission,fuel,brand,model,governorate
0,4672917,280000,2004,196122,Manual,Gas,Daewoo,Nubira 2,Cairo
1,6837266,700000,2011,170000,Automatic,Gas,Kia,Carens,ALSharqia
2,6867836,360000,2000,275000,Automatic,Gas,Skoda,Octavia A4,Ismailia
3,6881530,300000,2010,110000,Manual,Gas,Speranza,M11,Helwan
4,6884619,820000,2019,129000,Automatic,Gas,Subaru,Impreza,Cairo


,missing_values
listing_id,0
price,0
year,0
mileage_km,0
transmission,0
fuel,0
brand,0
model,0
governorate,0


In [6]:
# Reload and validate the saved artifact.
saved = pd.read_csv(OUTPUT_PATH)

assert len(saved) == len(preprocessed)
assert saved['listing_id'].is_unique
assert saved['price'].gt(0).all()
assert saved['price'].lt(MAX_PRICE_EGP).all()
assert saved['mileage_km'].between(0, MAX_MILEAGE_KM, inclusive='left').all()
assert saved['year'].between(MIN_YEAR, MAX_YEAR).all()
assert saved['governorate'].notna().all()
assert {'city', 'city_area'}.isdisjoint(saved.columns)

print('Validation passed: saved preprocessing data is internally consistent.')


Validation passed: saved preprocessing data is internally consistent.
